# 🧠 Smart Document Analyzer — Demo Notebook

This notebook walks through every component of the system end-to-end.

**What you'll see:**
1. PDF ingestion + text extraction
2. Text cleaning + layout detection
3. Smart chunking
4. Embedding generation
5. Vector store (FAISS + ChromaDB)
6. ML document classification
7. RAG query + answer generation
8. Answer quality evaluation
9. Response caching demo

---
> **Run each cell in order.** Make sure you have run `src/ml/train_pipeline.py` at least once before running this notebook.

## 0️⃣ Setup — paths and imports

In [17]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Make sure we're running from project root
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

print(f'Project root: {ROOT_DIR}')
print(f'Python: {sys.version.split()[0]}')

# Verify key modules are importable
modules_to_check = [
    'src.config.settings',
    'src.ingestion.pdf_loader',
    'src.preprocessing.text_cleaner',
    'src.chunking.text_chunker',
    'src.embeddings.embedder',
    'src.ml.predictor',
    'src.rag.rag_pipeline',
]

print('\nChecking modules:')
all_ok = True
for mod in modules_to_check:
    try:
        __import__(mod)
        print(f'  ✅ {mod}')
    except Exception as e:
        print(f'  ❌ {mod} — {e}')
        all_ok = False

print('\n✅ All good — ready to run!' if all_ok else '\n⚠️ Fix the errors above first.')

Project root: m:\Machine learning carrer\Projects\End-to-end Systems\Smart Document analyzer
Python: 3.11.9

Checking modules:
  ✅ src.config.settings
  ✅ src.ingestion.pdf_loader
  ✅ src.preprocessing.text_cleaner
  ✅ src.chunking.text_chunker
  ✅ src.embeddings.embedder
  ✅ src.ml.predictor
  ✅ src.rag.rag_pipeline

✅ All good — ready to run!


## 1️⃣ PDF Ingestion

In [33]:
DOCUMENTS_FOLDER = '../data/raw/documents'

In [34]:
from src.ingestion.pdf_loader import load_documents_from_folder

print(f'Documents folder: {DOCUMENTS_FOLDER}')

# Check what PDFs are available
if os.path.exists(DOCUMENTS_FOLDER):
    pdfs = [f for f in os.listdir(DOCUMENTS_FOLDER) if f.endswith('.pdf')]
    print(f'Found {len(pdfs)} PDF(s):')
    for p in pdfs:
        size = os.path.getsize(os.path.join(DOCUMENTS_FOLDER, p))
        print(f'  • {p}  ({size/1024:.1f} KB)')
else:
    print('⚠️  Documents folder not found.')
    print('Please upload a PDF via the Streamlit UI first, or place one in:')
    print(f'   {DOCUMENTS_FOLDER}')

Documents folder: ../data/raw/documents
Found 5 PDF(s):
  • affidavit-template.pdf  (75.3 KB)
  • Blank-Simple-Invoice.pdf  (502.4 KB)
  • data-science-roadmap.pdf  (2745.2 KB)
  • Mahmoud's CV.pdf  (119.2 KB)
  • Mazen_IslamCV.pdf  (62.1 KB)


In [35]:
# Load and extract text from all PDFs
raw_docs = load_documents_from_folder(DOCUMENTS_FOLDER)

print(f'\nLoaded {len(raw_docs)} document(s):')
for name, text in raw_docs.items():
    words = len(text.split())
    chars = len(text)
    print(f'\n📄 {name}')
    print(f'   Characters : {chars:,}')
    print(f'   Words      : {words:,}')
    print(f'   Preview    : {text[:200].strip()}...')

Loading: affidavit-template.pdf
  → Extracted 3275 characters

Loading: Blank-Simple-Invoice.pdf
  → Extracted 347 characters

Loading: data-science-roadmap.pdf
  → Extracted 12321 characters

Loading: Mahmoud's CV.pdf
  Page 2: no text found, running OCR...
  → Extracted 3872 characters

Loading: Mazen_IslamCV.pdf
  → Extracted 2340 characters


Loaded 5 document(s):

📄 affidavit-template.pdf
   Characters : 3,275
   Words      : 397
   Preview    : --- Page 1 ---
AFFIDAVIT OF EXECUTION 
______________________________________________________________________ 
 
 
On ____________ (Date) declared to us, the witnesses that foregoing instrument 
cons...

📄 Blank-Simple-Invoice.pdf
   Characters : 347
   Words      : 51
   Preview    : --- Page 1 ---
INVOICE 
BILL TO: 
Date of Service: 
Client Address:
Phone: 
Email: 
Notes & Special Instructions: 
- 
- 
Invoice Due By: 
Description 
Quantity 
Unit Cost 
Amount 
Product Total 
Desc...

📄 data-science-roadmap.pdf
   Characters : 12,321
   Wo

## 2️⃣ Text Cleaning

In [36]:
from src.preprocessing.text_cleaner import clean_documents

clean_docs = clean_documents(raw_docs)

print('Cleaning results:')
for name in raw_docs:
    before = len(raw_docs[name])
    after  = len(clean_docs[name])
    reduction = (1 - after/before) * 100 if before > 0 else 0
    print(f'  {name}')
    print(f'    Before : {before:,} chars')
    print(f'    After  : {after:,} chars  ({reduction:.1f}% noise removed)')

Cleaned: affidavit-template.pdf — 3234 chars
Cleaned: Blank-Simple-Invoice.pdf — 330 chars
Cleaned: data-science-roadmap.pdf — 11790 chars
Cleaned: Mahmoud's CV.pdf — 3800 chars
Cleaned: Mazen_IslamCV.pdf — 2309 chars
Cleaning results:
  affidavit-template.pdf
    Before : 3,275 chars
    After  : 3,234 chars  (1.3% noise removed)
  Blank-Simple-Invoice.pdf
    Before : 347 chars
    After  : 330 chars  (4.9% noise removed)
  data-science-roadmap.pdf
    Before : 12,321 chars
    After  : 11,790 chars  (4.3% noise removed)
  Mahmoud's CV.pdf
    Before : 3,872 chars
    After  : 3,800 chars  (1.9% noise removed)
  Mazen_IslamCV.pdf
    Before : 2,340 chars
    After  : 2,309 chars  (1.3% noise removed)


## 3️⃣ Layout Detection

In [37]:
from src.dl.layout_detector import detect_layouts_batch

layouts = detect_layouts_batch(clean_docs, pdf_folder=DOCUMENTS_FOLDER)

print('Layout detection results:')
for name, layout in layouts.items():
    summary = layout.summary()
    print(f'\n📄 {name}')
    print(f'   Method     : {summary["method"]}')
    print(f'   Pages      : {summary["pages"]}')
    print(f'   Regions    : {summary["regions"]}')
    print(f'   Breakdown  : {summary["breakdown"]}')

    print('\n   First 5 regions:')
    for r in layout.regions[:5]:
        print(f'     [{r.region_type:12s}] pg{r.page}  {r.text[:80]}...')


  Layout detection: affidavit-template.pdf
  ✅ affidavit-template.pdf: 30 regions {'title': 18, 'paragraph': 12}

  Layout detection: Blank-Simple-Invoice.pdf
  ✅ Blank-Simple-Invoice.pdf: 25 regions {'title': 25}

  Layout detection: data-science-roadmap.pdf
  ✅ data-science-roadmap.pdf: 187 regions {'title': 127, 'paragraph': 59, 'list': 1}

  Layout detection: Mahmoud's CV.pdf
  ✅ Mahmoud's CV.pdf: 41 regions {'table': 5, 'title': 16, 'paragraph': 20}

  Layout detection: Mazen_IslamCV.pdf
  ✅ Mazen_IslamCV.pdf: 33 regions {'title': 23, 'paragraph': 9, 'table': 1}
Layout detection results:

📄 affidavit-template.pdf
   Method     : rules
   Pages      : 1
   Regions    : 30
   Breakdown  : {'title': 18, 'paragraph': 12}

   First 5 regions:
     [title       ] pg1  AFFIDAVIT OF EXECUTION...
     [paragraph   ] pg1  On ____________ (Date) declared to us, the witnesses that foregoing instrument c...
     [paragraph   ] pg1  We, the witnesses are all present at the specified date and t

## 4️⃣ Smart Chunking

In [38]:
from src.dl.layout_chunker import chunk_documents_by_layout
from src.config.settings import CHUNK_SIZE

chunked_docs = chunk_documents_by_layout(layouts, chunk_size=CHUNK_SIZE)

print(f'Chunk size setting: {CHUNK_SIZE} words\n')

all_chunks = []
for name, chunks in chunked_docs.items():
    print(f'📄 {name}  →  {len(chunks)} chunks')
    all_chunks.extend(chunks)

    # Show distribution of chunk types
    from collections import Counter
    types = Counter(c.get('region_type', 'unknown') for c in chunks)
    print(f'   Types: {dict(types)}')

    # Show sample chunks
    print('\n   Sample chunks:')
    for i, chunk in enumerate(chunks[:3]):
        words = len(chunk['text'].split())
        print(f'     Chunk {i+1} [{chunk["region_type"]}] pg{chunk["page"]}'
              f'  {words}w  "{chunk["text"][:80]}..."')

print(f'\nTotal chunks across all documents: {len(all_chunks)}')


  Chunking by layout: affidavit-template.pdf
  Layout chunker: 4 chunks from 30 regions

  Chunking by layout: Blank-Simple-Invoice.pdf
  Layout chunker: 0 chunks from 25 regions

  Chunking by layout: data-science-roadmap.pdf
  Layout chunker: 58 chunks from 187 regions

  Chunking by layout: Mahmoud's CV.pdf
  Layout chunker: 14 chunks from 41 regions

  Chunking by layout: Mazen_IslamCV.pdf
  Layout chunker: 10 chunks from 33 regions
Chunk size setting: 200 words

📄 affidavit-template.pdf  →  4 chunks
   Types: {'section': 4}

   Sample chunks:
     Chunk 1 [section] pg1  117w  "AFFIDAVIT OF EXECUTION
On ____________ (Date) declared to us, the witnesses that..."
     Chunk 2 [section] pg1  39w  "Testator
is over eighteen (18) years of age and of sound mind. We, the witnesses..."
     Chunk 3 [section] pg1  59w  "SWORN STATEMENT
Under penalty of perjury, we hereby declare and affirm that the ..."
📄 Blank-Simple-Invoice.pdf  →  0 chunks
   Types: {}

   Sample chunks:
📄 data-science-

## 5️⃣ Embeddings

In [39]:
import numpy as np
from src.embeddings.embedder import embed_chunks, embed_query

chunk_texts = [c['text'] for c in all_chunks]

print(f'Embedding {len(chunk_texts)} chunks...')
embeddings = embed_chunks(chunk_texts)

print(f'\nEmbedding matrix shape : {embeddings.shape}')
print(f'Embedding dimensions   : {embeddings.shape[1]}')
print(f'Memory usage           : {embeddings.nbytes / 1024:.1f} KB')
print(f'Value range            : [{embeddings.min():.3f}, {embeddings.max():.3f}]')

# Test query embedding
test_query = 'What is this document about?'
q_vec = embed_query(test_query)
print(f'\nQuery embedding shape  : {q_vec.shape}')
print(f'Query: "{test_query}"')

Embedding 86 chunks...


Batches: 100%|██████████| 3/3 [00:01<00:00,  1.70it/s]


Embedding matrix shape : (86, 384)
Embedding dimensions   : 384
Memory usage           : 129.0 KB
Value range            : [-0.197, 0.203]

Query embedding shape  : (1, 384)
Query: "What is this document about?"


## 6️⃣ ML Document Classification

In [40]:
from src.ml.predictor import predict_document_type

print('Document classification results:\n')

for name, text in clean_docs.items():
    result = predict_document_type(text)

    label  = result['label']
    conf   = result['confidence']
    scores = result['all_scores']

    icons = {'contract': '🟦', 'invoice': '🟨', 'research': '🟩', 'unknown': '⬜'}
    icon  = icons.get(label, '❓')

    print(f'📄 {name}')
    print(f'   {icon} Predicted : {label}  (confidence: {conf:.1%})')
    print(f'   All scores:')
    for cls, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
        bar = '█' * int(score * 20) + '░' * (20 - int(score * 20))
        print(f'     {cls:10s} {bar} {score:.1%}')
    print()

Document classification results:

📄 affidavit-template.pdf
   🟦 Predicted : contract  (confidence: 99.7%)
   All scores:
     contract   ███████████████████░ 99.7%
     research   ░░░░░░░░░░░░░░░░░░░░ 0.3%
     invoice    ░░░░░░░░░░░░░░░░░░░░ 0.0%

📄 Blank-Simple-Invoice.pdf
   🟨 Predicted : invoice  (confidence: 92.0%)
   All scores:
     invoice    ██████████████████░░ 92.0%
     research   █░░░░░░░░░░░░░░░░░░░ 7.8%
     contract   ░░░░░░░░░░░░░░░░░░░░ 0.1%

📄 data-science-roadmap.pdf
   🟨 Predicted : invoice  (confidence: 66.8%)
   All scores:
     invoice    █████████████░░░░░░░ 66.8%
     research   ██████░░░░░░░░░░░░░░ 33.0%
     contract   ░░░░░░░░░░░░░░░░░░░░ 0.2%

📄 Mahmoud's CV.pdf
   🟨 Predicted : invoice  (confidence: 87.7%)
   All scores:
     invoice    █████████████████░░░ 87.7%
     research   ██░░░░░░░░░░░░░░░░░░ 12.3%
     contract   ░░░░░░░░░░░░░░░░░░░░ 0.0%

📄 Mazen_IslamCV.pdf
   🟩 Predicted : research  (confidence: 52.7%)
   All scores:
     research   ██████████░

## 7️⃣ Vector Store — FAISS

In [41]:
from src.vector_db.faiss_store import (
    build_faiss_index, save_faiss_index, load_faiss_index, search_faiss
)

# Build metadata
metadata = [
    {
        'source':      c['source'],
        'text':        c['text'],
        'region_type': c.get('region_type', 'paragraph'),
        'page':        c.get('page', 0)
    }
    for c in all_chunks
]

# Build and save index
index = build_faiss_index(embeddings)
save_faiss_index(index, metadata)

print(f'FAISS index: {index.ntotal} vectors, dim={embeddings.shape[1]}')

# Test retrieval
test_query = 'What is the main topic of this document?'
q_vec      = embed_query(test_query)

results = search_faiss(index, metadata, q_vec, top_k=3)
print(f'\nTest query: "{test_query}"')
print(f'Top 3 results:')
for i, r in enumerate(results, 1):
    print(f'\n  [{i}] Score: {r["score"]:.4f}  Source: {r["source"]}  Page: {r.get("page","?")}')
    print(f'      {r["text"][:150]}...')

FAISS index built — 86 vectors, dim=384
Saved FAISS index → vector_db/faiss_index/index.faiss
FAISS index: 86 vectors, dim=384

Test query: "What is the main topic of this document?"
Top 3 results:

  [1] Score: 0.2478  Source: data-science-roadmap.pdf  Page: ?
      Essential Concepts
Natural Language Processing (NLP) Text preprocessing (tokenization, stemming, lemmatization) Sentiment analysis Named entity recogn...

  [2] Score: 0.2436  Source: affidavit-template.pdf  Page: ?
      AFFIDAVIT OF EXECUTION
On ____________ (Date) declared to us, the witnesses that foregoing instrument consisting of ___ (number of pages), including t...

  [3] Score: 0.2348  Source: Mahmoud's CV.pdf  Page: ?
      Benha University
Relevant Coursework: python , math ,probability and statistics , machine learning , neural networks , Deep learning, fundamentals of ...


## 8️⃣ RAG Pipeline — Full Question Answering

In [42]:
from src.rag.rag_pipeline import ask

# Test questions — change these to match your document
test_questions = [
    'What is the main topic of this document?',
    'Who are the key people or parties mentioned?',
    'What are the most important points?',
]

print('RAG Question Answering Demo')
print('=' * 55)

for question in test_questions:
    print(f'\nQ: {question}')
    result = ask(
        question=question,
        top_k=5,
        store='faiss',
        use_cache=False,       # disable cache for demo
        use_expansion=True
    )

    print(f'A: {result["answer"]}')
    print(f'\n   Doc type  : {result["doc_type"]}')
    print(f'   Latency   : {result["latency_s"]}s')
    print(f'   Sources   : {[s["source"] for s in result["sources"]]}')
    print(f'   Expanded  : {result["expanded_queries"]}')
    print('-' * 55)

RAG Question Answering Demo

Q: What is the main topic of this document?
  Query expanded (LLM): 3 variants
  Retrieving from: FAISS (top_k=5)
  Retrieving from: FAISS (top_k=5)
  Retrieving from: FAISS (top_k=5)


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.50it/s]


A: I could not find this information in the document.

   Doc type  : unknown
   Latency   : 1.23s
   Sources   : ['data-science-roadmap.pdf', 'affidavit-template.pdf', "Mahmoud's CV.pdf", "Mahmoud's CV.pdf", 'data-science-roadmap.pdf']
   Expanded  : ['What is the main topic of this document?', 'What is the primary subject of this document?', 'What is the central theme of this document?']
-------------------------------------------------------

Q: Who are the key people or parties mentioned?
  Query expanded (LLM): 3 variants
  Retrieving from: FAISS (top_k=5)
  Retrieving from: FAISS (top_k=5)
  Retrieving from: FAISS (top_k=5)


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.74it/s]


A: Mahmoud Amr Amin is the key person mentioned.

   Doc type  : unknown
   Latency   : 1.25s
   Sources   : ["Mahmoud's CV.pdf", 'data-science-roadmap.pdf', 'data-science-roadmap.pdf', 'data-science-roadmap.pdf', 'data-science-roadmap.pdf']
   Expanded  : ['Who are the key people or parties mentioned?', 'Who are the main individuals or groups mentioned?', 'What are the key figures or entities referred to?']
-------------------------------------------------------

Q: What are the most important points?
  Query expanded (LLM): 3 variants
  Retrieving from: FAISS (top_k=5)
  Retrieving from: FAISS (top_k=5)
  Retrieving from: FAISS (top_k=5)


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.81it/s]

A: I could not find this information in the document.

   Doc type  : unknown
   Latency   : 1.26s
   Sources   : ['affidavit-template.pdf', 'data-science-roadmap.pdf', 'data-science-roadmap.pdf', 'affidavit-template.pdf', 'data-science-roadmap.pdf']
   Expanded  : ['What are the most important points?', 'Key takeaways from the document', 'Main points discussed in the text']
-------------------------------------------------------


## 9️⃣ Answer Quality Evaluation

In [43]:
from src.utils.evaluator import evaluate_answer

# Use last result from previous cell
question = test_questions[0]
result   = ask(question, top_k=5, use_cache=False)
metrics  = evaluate_answer(question, result['answer'], result['chunks'])

print(f'Answer Quality Evaluation')
print(f'=' * 40)
print(f'Question  : {question}')
print(f'Answer    : {result["answer"][:150]}...')
print()

labels = [
    ('🔍 Retrieval Relevance', 'retrieval_relevance'),
    ('📎 Answer Grounding',    'answer_grounding'),
    ('🏆 Top Chunk Score',     'top_chunk_score'),
    ('📊 Confidence',          'confidence'),
]

for label, key in labels:
    val = metrics.get(key, 0)
    bar = '█' * int(val * 20) + '░' * (20 - int(val * 20))
    print(f'{label:30s} {bar} {val:.1%}')

print(f'\nOverall quality : {metrics["quality_label"]}')

print(f'\nLatency breakdown:')
for k, v in result.get('timing', {}).items():
    print(f'  {k:20s} : {v}s')

  Query expanded (LLM): 3 variants
  Retrieving from: FAISS (top_k=5)
  Retrieving from: FAISS (top_k=5)
  Retrieving from: FAISS (top_k=5)


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.85it/s]

Answer Quality Evaluation
Question  : What is the main topic of this document?
Answer    : I could not find this information in the document....

🔍 Retrieval Relevance          ████░░░░░░░░░░░░░░░░ 22.1%
📎 Answer Grounding             ░░░░░░░░░░░░░░░░░░░░ 0.0%
🏆 Top Chunk Score              ████░░░░░░░░░░░░░░░░ 24.8%
📊 Confidence                   ███░░░░░░░░░░░░░░░░░ 15.0%

Overall quality : 🔴 Poor

Latency breakdown:
  expansion_s          : 0.571s
  retrieval_s          : 0.037s
  generation_s         : 0.592s
  eval_s               : 0.171s
  total_s              : 1.37s


## 🔟 Response Cache Demo

In [44]:
import time
from src.utils.cache import get_cache, clear_cache

# Clear cache to start fresh
clear_cache()

question = 'What is the main topic of this document?'

# First call — hits the LLM
print('First call (no cache):')
t0      = time.time()
result1 = ask(question, use_cache=True, use_expansion=False)
t1      = time.time()
print(f'  Latency   : {t1-t0:.2f}s')
print(f'  From cache: {result1["from_cache"]}')
print(f'  Answer    : {result1["answer"][:100]}...')

print()

# Second call — same question, served from cache
print('Second call (cached):')
t0      = time.time()
result2 = ask(question, use_cache=True, use_expansion=False)
t1      = time.time()
print(f'  Latency   : {t1-t0:.4f}s')
print(f'  From cache: {result2["from_cache"]}')
print(f'  Answer    : {result2["answer"][:100]}...')

print()

# Semantic similarity test — slightly different phrasing
similar_q = 'What does this document cover?'
print(f'Semantic match test: "{similar_q}"')
t0      = time.time()
result3 = ask(similar_q, use_cache=True, use_expansion=False)
t1      = time.time()
print(f'  Latency   : {t1-t0:.4f}s')
print(f'  From cache: {result3["from_cache"]}')
if result3.get('cache_sim'):
    print(f'  Similarity: {result3["cache_sim"]}')

cache = get_cache()
print(f'\nCache entries stored: {cache.size}')

  Cache cleared.
First call (no cache):
  Retrieving from: FAISS (top_k=5)


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.69it/s]


  💾 Cached: What is the main topic of this document?
  Latency   : 0.78s
  From cache: False
  Answer    : I could not find this information in the document....

Second call (cached):
  ✅ Cache HIT (exact): What is the main topic of this document?
  Latency   : 0.0000s
  From cache: True
  Answer    : I could not find this information in the document....

Semantic match test: "What does this document cover?"
  Retrieving from: FAISS (top_k=5)


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.10it/s]

  💾 Cached: What does this document cover?
  Latency   : 0.7768s
  From cache: False

Cache entries stored: 2


## 1️⃣1️⃣ Query Expansion Demo

In [45]:
from src.rag.query_expander import expand_query

test_queries = [
    'What is the salary?',
    'Who wrote this document?',
    'What are the main findings?',
    'When is the payment due?',
]

print('Query Expansion Demo')
print('=' * 50)

for q in test_queries:
    expanded = expand_query(q, use_llm=True)
    print(f'\nOriginal : {q}')
    for i, variant in enumerate(expanded):
        tag = 'Original' if i == 0 else f'Variant {i}'
        print(f'  {tag:12s}: {variant}')

Query Expansion Demo
  Query expanded (LLM): 3 variants

Original : What is the salary?
  Original    : What is the salary?
  Variant 1   : What is the average salary?
  Variant 2   : What compensation does the job offer?
  Query expanded (LLM): 3 variants

Original : Who wrote this document?
  Original    : Who wrote this document?
  Variant 1   : Who is the author of this document?
  Variant 2   : Who composed this document?
  Query expanded (LLM): 3 variants

Original : What are the main findings?
  Original    : What are the main findings?
  Variant 1   : What are the key results?
  Variant 2   : What were the primary conclusions?
  Query expanded (LLM): 3 variants

Original : When is the payment due?
  Original    : When is the payment due?
  Variant 1   : When is the payment deadline?
  Variant 2   : What is the payment due date?


## 1️⃣2️⃣ Full System Summary

Run this cell last for a complete system health report.

In [46]:
import os
from src.config.settings import (
    ROOT_DIR, FAISS_INDEX_PATH,
    ML_MODEL_PATH, CHUNK_SIZE, TOP_K, GROQ_MODEL
)

print('=' * 55)
print('  🧠 Smart Document Analyzer — System Report')
print('=' * 55)
checks = [
    ('ML model trained',    os.path.exists(ML_MODEL_PATH)),
    ('FAISS index built',   os.path.exists("../vector_db/faiss_index/index.faiss")),
    ('Documents folder',    os.path.exists(DOCUMENTS_FOLDER)),
    ('Groq API key set',    bool(os.getenv('GROQ_API_KEY', ''))),
]

print('\nSystem checks:')
for label, status in checks:
    icon = '✅' if status else '❌'
    print(f'  {icon}  {label}')

print('\nConfiguration:')
print(f'  Chunk size      : {CHUNK_SIZE} words')
print(f'  Top-K retrieval : {TOP_K}')
print(f'  LLM model       : {GROQ_MODEL}')

# Count documents
if os.path.exists(DOCUMENTS_FOLDER):
    pdfs = [f for f in os.listdir(DOCUMENTS_FOLDER) if f.endswith('.pdf')]
    print(f'  PDFs loaded     : {len(pdfs)}')

# Cache stats
from src.utils.cache import get_cache
cache = get_cache()
print(f'  Cache entries   : {cache.size}')

print('\n' + '=' * 55)
print('  ✅ Demo complete!')
print('  Run: streamlit run "front end/app.py" for the full UI')
print('=' * 55)

  🧠 Smart Document Analyzer — System Report

System checks:
  ✅  ML model trained
  ✅  FAISS index built
  ✅  Documents folder
  ✅  Groq API key set

Configuration:
  Chunk size      : 200 words
  Top-K retrieval : 5
  LLM model       : llama-3.1-8b-instant
  PDFs loaded     : 5
  Cache entries   : 2

  ✅ Demo complete!
  Run: streamlit run "front end/app.py" for the full UI
